<a href="https://colab.research.google.com/github/0906Bao/MedicalPPE_yolo/blob/main/MPPE_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')


yaml_path = '/content/drive/MyDrive/MedicalPPE/Data/Main/data.yaml'


print("Bắt đầu huấn luyện YOLOv8...")
results = model.train(data=yaml_path,
    epochs=50,
    device=0,
    imgsz=640,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    flipud=0.5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

csv_files = glob.glob('/content/runs/detect/*/results.csv')
if len(csv_files) == 0:
    print("Không tìm thấy file results.csv!")
else:
    latest_csv = max(csv_files, key=os.path.getctime)
    df = pd.read_csv(latest_csv)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    if 'metrics/mAP50(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='Validation mAP50', color='green', marker='o')
    if 'metrics/mAP50-95(B)' in df.columns:
        plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Validation mAP50-95', color='limegreen', linestyle='--')
    plt.title('Biểu đồ Độ chính xác (mAP)')
    plt.xlabel('Epoch')
    plt.ylabel('Độ chính xác')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue', marker='o')
    plt.plot(df['epoch'], df['val/box_loss'], label='Validation Box Loss', color='orange', marker='o')
    plt.title('So sánh Loss (Train vs Validation)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
from ultralytics import YOLO
import glob
import os
import cv2
from google.colab.patches import cv2_imshow

weight_files = glob.glob('/content/runs/detect/*/weights/best.pt')
if len(weight_files) == 0:
    print("Chưa tìm thấy file mô hình huấn luyện! Đang dùng mô hình gốc yolov8s.pt")
    model = YOLO('yolov8s.pt')
else:
    latest_weight = max(weight_files, key=os.path.getctime)
    print(f"Đã tải mô hình tối ưu nhất tại: {latest_weight}")
    model = YOLO(latest_weight)

valid_image_folder = '/content/drive/MyDrive/MedicalPPE/Data/Main/valid/images/*.jpg'

image_paths = sorted(glob.glob(valid_image_folder))

if len(image_paths) == 0:
    print("Không tìm thấy ảnh nào trong thư mục Validation. Vui lòng kiểm tra lại đường dẫn!")
else:
    num_images_to_show = min(10, len(image_paths))
    selected_images = image_paths[:num_images_to_show]
    print(f"Tìm thấy tổng cộng {len(image_paths)} ảnh. Đang tiến hành nhận diện {num_images_to_show} ảnh đầu tiên...\n")

    for i, img_path in enumerate(selected_images):
        print(f"--- Ảnh {i+1}/{num_images_to_show}: {os.path.basename(img_path)} ---")

        results = model.predict(source=img_path, conf=0.25, save=True, verbose=False)

        predict_folders = glob.glob('/content/runs/detect/predict*')
        latest_predict_folder = max(predict_folders, key=os.path.getctime)

        result_image_path = os.path.join(latest_predict_folder, os.path.basename(img_path))

        if os.path.exists(result_image_path):
            img = cv2.imread(result_image_path)
            cv2_imshow(img)
        else:

            fallback_images = glob.glob(f"{latest_predict_folder}/*")
            if len(fallback_images) > 0:
                img = cv2.imread(fallback_images[-1])
                cv2_imshow(img)

        print("\n" + "="*50 + "\n")